## Metric policy used here

This rebuild follows one consistent rule:

- use the **official ACS field directly** when the subject table already exposes the exact percent, count, median, mean, or rate needed
- derive a metric only when the final metric is a bundle or complement, for example:
  - `pct_less_than_high_school = 100 - pct_high_school_or_higher`
  - `pct_some_college_or_associate = pct_some_college_no_degree + pct_associates_degree`
  - grouped income shares such as `<25k`, `25k–50k`, `50k–100k`, and `100k+`
- keep lineage for every final metric in a CSV output


In [1]:
from __future__ import annotations

import os
from pathlib import Path
import json

import pandas as pd
from dotenv import load_dotenv

try:
    from sqlalchemy import create_engine
    SQLALCHEMY_AVAILABLE = True
except Exception:
    SQLALCHEMY_AVAILABLE = False

import psycopg2


In [2]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for p in candidates:
        if (p / '.env').exists():
            return p
    return cwd

PROJECT_ROOT = find_project_root()
ENV_FILE = PROJECT_ROOT / '.env'

if not ENV_FILE.exists():
    raise FileNotFoundError(f'.env file not found at: {ENV_FILE}')

load_dotenv(ENV_FILE)
print(f'[ENV] Loaded .env from: {ENV_FILE}')

DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_SCHEMA = os.getenv('DB_SCHEMA', 'public')

missing = [k for k, v in {
    'DB_HOST': DB_HOST,
    'DB_PORT': DB_PORT,
    'DB_NAME': DB_NAME,
    'DB_USER': DB_USER,
    'DB_PASSWORD': DB_PASSWORD,
}.items() if v in {None, ''}]
if missing:
    raise ValueError(f'Missing DB config values in .env: {missing}')

print(f'[DB] Target database: {DB_NAME} on {DB_HOST}:{DB_PORT}')
print(f'[DB] Target schema: {DB_SCHEMA}')


[ENV] Loaded .env from: D:\Projects\Community-Pulse\.env
[DB] Target database: mydb on localhost:5432
[DB] Target schema: public


In [3]:
def get_sqlalchemy_engine():
    if not SQLALCHEMY_AVAILABLE:
        return None
    url = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
    return create_engine(url)

def get_psycopg2_conn():
    return psycopg2.connect(
        host=DB_HOST,
        port=int(DB_PORT),
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
    )

ENGINE = get_sqlalchemy_engine()
CONN = get_psycopg2_conn()
print('[OK] Database connection established')


[OK] Database connection established


In [4]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'acs' / 'analysis' / 'base'
SQL_DIR = PROJECT_ROOT / 'sql' / 'acs' / 'fact'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SQL_DIR.mkdir(parents=True, exist_ok=True)

print(f'[OK] Output dir: {OUTPUT_DIR}')
print(f'[OK] SQL dir: {SQL_DIR}')


[OK] Output dir: D:\Projects\Community-Pulse\outputs\acs\analysis\base
[OK] SQL dir: D:\Projects\Community-Pulse\sql\acs\fact


In [5]:
def read_sql_df(query: str) -> pd.DataFrame:
    if ENGINE is not None:
        return pd.read_sql(query, ENGINE)
    with CONN.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)

def execute_sql(sql_text: str) -> None:
    with CONN.cursor() as cur:
        cur.execute(sql_text)
    CONN.commit()

def relation_exists(relation_name: str, schema: str = DB_SCHEMA) -> bool:
    q = '''
    SELECT 1
    FROM information_schema.views
    WHERE table_schema = %s AND table_name = %s
    UNION ALL
    SELECT 1
    FROM information_schema.tables
    WHERE table_schema = %s AND table_name = %s
    LIMIT 1
    '''
    with CONN.cursor() as cur:
        cur.execute(q, (schema, relation_name, schema, relation_name))
        return cur.fetchone() is not None

def get_columns(relation_name: str, schema: str = DB_SCHEMA) -> list[str]:
    q = '''
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = %s AND table_name = %s
    ORDER BY ordinal_position
    '''
    with CONN.cursor() as cur:
        cur.execute(q, (schema, relation_name))
        rows = cur.fetchall()
    return [r[0] for r in rows]


In [6]:
BASE_RELATION = 'fact_acs_tract_profile'
SOURCE_VIEWS = {
    's0101': 'vw_int_acs_s0101_all_years',
    's1101': 'vw_int_acs_s1101_all_years',
    's1501': 'vw_int_acs_s1501_all_years',
    's1701': 'vw_int_acs_s1701_all_years',
    's1901': 'vw_int_acs_s1901_all_years',
    's2301': 'vw_int_acs_s2301_all_years',
    's1601': 'vw_int_acs_s1601_all_years',
}

required_relations = [BASE_RELATION] + list(SOURCE_VIEWS.values())
missing_relations = [r for r in required_relations if not relation_exists(r, DB_SCHEMA)]
if missing_relations:
    raise ValueError(f'Missing required tables/views: {missing_relations}')

for rel in required_relations:
    print(f'[OK] Found: {DB_SCHEMA}.{rel}')


[OK] Found: public.fact_acs_tract_profile
[OK] Found: public.vw_int_acs_s0101_all_years
[OK] Found: public.vw_int_acs_s1101_all_years
[OK] Found: public.vw_int_acs_s1501_all_years
[OK] Found: public.vw_int_acs_s1701_all_years
[OK] Found: public.vw_int_acs_s1901_all_years
[OK] Found: public.vw_int_acs_s2301_all_years
[OK] Found: public.vw_int_acs_s1601_all_years


In [7]:
metric_specs = [
    {'metric_name':'age_under_18_population','category':'age','sql_expr':'s0101.s0101_c01_022e','source_view':SOURCE_VIEWS['s0101'],'source_columns':['s0101_c01_022e'],'definition_policy':'official_field','numerator':'population under 18','denominator':None,'include_in_cluster_input':False,'notes':'Official selected age category count from S0101'},
    {'metric_name':'age_18_24_population','category':'age','sql_expr':'s0101.s0101_c01_023e','source_view':SOURCE_VIEWS['s0101'],'source_columns':['s0101_c01_023e'],'definition_policy':'official_field','numerator':'population age 18 to 24','denominator':None,'include_in_cluster_input':False,'notes':'Official selected age category count from S0101'},
    {'metric_name':'age_65_plus_population','category':'age','sql_expr':'s0101.s0101_c01_030e','source_view':SOURCE_VIEWS['s0101'],'source_columns':['s0101_c01_030e'],'definition_policy':'official_field','numerator':'population age 65 and over','denominator':None,'include_in_cluster_input':False,'notes':'Official selected age category count from S0101'},
    {'metric_name':'pct_age_under_18','category':'age','sql_expr':'s0101.s0101_c02_022e','source_view':SOURCE_VIEWS['s0101'],'source_columns':['s0101_c02_022e'],'definition_policy':'official_field','numerator':'population under 18','denominator':'total population','include_in_cluster_input':True,'notes':'Official selected age category percent from S0101'},
    {'metric_name':'pct_age_18_24','category':'age','sql_expr':'s0101.s0101_c02_023e','source_view':SOURCE_VIEWS['s0101'],'source_columns':['s0101_c02_023e'],'definition_policy':'official_field','numerator':'population age 18 to 24','denominator':'total population','include_in_cluster_input':True,'notes':'Official selected age category percent from S0101'},
    {'metric_name':'pct_age_65_plus','category':'age','sql_expr':'s0101.s0101_c02_030e','source_view':SOURCE_VIEWS['s0101'],'source_columns':['s0101_c02_030e'],'definition_policy':'official_field','numerator':'population age 65 and over','denominator':'total population','include_in_cluster_input':True,'notes':'Official selected age category percent from S0101'},
    {'metric_name':'median_age_years','category':'age','sql_expr':'s0101.s0101_c01_032e','source_view':SOURCE_VIEWS['s0101'],'source_columns':['s0101_c01_032e'],'definition_policy':'official_field','numerator':'median age','denominator':None,'include_in_cluster_input':False,'notes':'Official summary indicator from S0101'},

    {'metric_name':'total_households_s1101','category':'household_composition','sql_expr':'s1101.s1101_c01_001e','source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_001e'],'definition_policy':'official_field','numerator':'total households','denominator':None,'include_in_cluster_input':False,'notes':'Official total households from S1101'},
    {'metric_name':'avg_household_size_v2','category':'household_composition','sql_expr':'s1101.s1101_c01_002e','source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_002e'],'definition_policy':'official_field','numerator':'average household size','denominator':None,'include_in_cluster_input':True,'notes':'Official average household size from S1101'},
    {'metric_name':'total_families_s1101','category':'household_composition','sql_expr':'s1101.s1101_c01_003e','source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_003e'],'definition_policy':'official_field','numerator':'total families','denominator':None,'include_in_cluster_input':False,'notes':'Official family count from S1101'},
    {'metric_name':'pct_family_households','category':'household_composition','sql_expr':"CASE WHEN s1101.s1101_c01_001e > 0 THEN ROUND(100.0 * s1101.s1101_c01_003e / s1101.s1101_c01_001e, 2) ELSE NULL END",'source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_003e','s1101_c01_001e'],'definition_policy':'derived_from_counts','numerator':'total families','denominator':'total households','include_in_cluster_input':True,'notes':'Derived family-household share from S1101 counts'},
    {'metric_name':'pct_nonfamily_households','category':'household_composition','sql_expr':"CASE WHEN s1101.s1101_c01_001e > 0 THEN ROUND(100.0 - (100.0 * s1101.s1101_c01_003e / s1101.s1101_c01_001e), 2) ELSE NULL END",'source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_003e','s1101_c01_001e'],'definition_policy':'derived_from_counts','numerator':'nonfamily households','denominator':'total households','include_in_cluster_input':False,'notes':'Complement of family-household share'},
    {'metric_name':'households_with_own_children_under_18','category':'household_composition','sql_expr':'s1101.s1101_c01_005e','source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_005e'],'definition_policy':'official_field','numerator':'households with own children under 18','denominator':None,'include_in_cluster_input':False,'notes':'Official count from S1101'},
    {'metric_name':'pct_households_with_own_children_under_18','category':'household_composition','sql_expr':"CASE WHEN s1101.s1101_c01_001e > 0 THEN ROUND(100.0 * s1101.s1101_c01_005e / s1101.s1101_c01_001e, 2) ELSE NULL END",'source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_005e','s1101_c01_001e'],'definition_policy':'derived_from_counts','numerator':'households with own children under 18','denominator':'total households','include_in_cluster_input':True,'notes':'Derived from S1101 own-children count and total households'},
    {'metric_name':'pct_households_with_under_18','category':'household_composition','sql_expr':'s1101.s1101_c01_010e','source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_010e'],'definition_policy':'official_field','numerator':'households with one or more people under 18','denominator':'total households','include_in_cluster_input':True,'notes':'Official selected-household-type percentage from S1101'},
    {'metric_name':'pct_households_with_60_plus','category':'household_composition','sql_expr':'s1101.s1101_c01_011e','source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_011e'],'definition_policy':'official_field','numerator':'households with one or more people 60 years and over','denominator':'total households','include_in_cluster_input':False,'notes':'Official selected-household-type percentage from S1101'},
    {'metric_name':'pct_households_with_65_plus','category':'household_composition','sql_expr':'s1101.s1101_c01_012e','source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_012e'],'definition_policy':'official_field','numerator':'households with one or more people 65 years and over','denominator':'total households','include_in_cluster_input':True,'notes':'Official selected-household-type percentage from S1101'},
    {'metric_name':'pct_one_person_households','category':'household_composition','sql_expr':'s1101.s1101_c01_013e','source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_013e'],'definition_policy':'official_field','numerator':'households with householder living alone','denominator':'total households','include_in_cluster_input':True,'notes':'Official selected-household-type percentage from S1101'},
    {'metric_name':'pct_senior_living_alone_households','category':'household_composition','sql_expr':'s1101.s1101_c01_014e','source_view':SOURCE_VIEWS['s1101'],'source_columns':['s1101_c01_014e'],'definition_policy':'official_field','numerator':'households with householder living alone age 65+','denominator':'total households','include_in_cluster_input':False,'notes':'Official selected-household-type percentage from S1101'},

    {'metric_name':'education_population_25_plus','category':'education','sql_expr':'s1501.s1501_c01_006e','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c01_006e'],'definition_policy':'official_field','numerator':'population 25 years and over','denominator':None,'include_in_cluster_input':False,'notes':'Official 25+ education base from S1501'},
    {'metric_name':'pct_less_than_high_school','category':'education','sql_expr':'CASE WHEN s1501.s1501_c02_014e IS NOT NULL THEN ROUND(100.0 - s1501.s1501_c02_014e, 2) ELSE NULL END','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c02_014e'],'definition_policy':'derived_from_official_fields','numerator':'population 25+ with less than high school','denominator':'population 25 years and over','include_in_cluster_input':True,'notes':'Derived as complement of official high-school-or-higher percentage from S1501'},
    {'metric_name':'pct_high_school_grad','category':'education','sql_expr':'s1501.s1501_c02_009e','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c02_009e'],'definition_policy':'official_field','numerator':'population 25+ high school graduate','denominator':'population 25 years and over','include_in_cluster_input':True,'notes':'Official S1501 percentage'},
    {'metric_name':'pct_some_college_no_degree','category':'education','sql_expr':'s1501.s1501_c02_010e','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c02_010e'],'definition_policy':'official_field','numerator':'population 25+ some college no degree','denominator':'population 25 years and over','include_in_cluster_input':False,'notes':'Official S1501 percentage'},
    {'metric_name':'pct_associates_degree','category':'education','sql_expr':'s1501.s1501_c02_011e','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c02_011e'],'definition_policy':'official_field','numerator':'population 25+ associate degree','denominator':'population 25 years and over','include_in_cluster_input':False,'notes':'Official S1501 percentage'},
    {'metric_name':'pct_some_college_or_associate','category':'education','sql_expr':'CASE WHEN s1501.s1501_c02_010e IS NOT NULL OR s1501.s1501_c02_011e IS NOT NULL THEN ROUND(COALESCE(s1501.s1501_c02_010e, 0) + COALESCE(s1501.s1501_c02_011e, 0), 2) ELSE NULL END','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c02_010e','s1501_c02_011e'],'definition_policy':'derived_from_official_fields','numerator':'population 25+ some college or associate degree','denominator':'population 25 years and over','include_in_cluster_input':True,'notes':'Bundle of official S1501 percentages'},
    {'metric_name':'pct_bachelors_degree','category':'education','sql_expr':'s1501.s1501_c02_012e','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c02_012e'],'definition_policy':'official_field','numerator':'population 25+ bachelor degree','denominator':'population 25 years and over','include_in_cluster_input':False,'notes':'Official S1501 percentage'},
    {'metric_name':'pct_graduate_or_professional','category':'education','sql_expr':'s1501.s1501_c02_013e','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c02_013e'],'definition_policy':'official_field','numerator':'population 25+ graduate or professional degree','denominator':'population 25 years and over','include_in_cluster_input':False,'notes':'Official S1501 percentage'},
    {'metric_name':'pct_high_school_or_higher','category':'education','sql_expr':'s1501.s1501_c02_014e','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c02_014e'],'definition_policy':'official_field','numerator':'population 25+ high school or higher','denominator':'population 25 years and over','include_in_cluster_input':False,'notes':'Official S1501 percentage'},
    {'metric_name':'pct_bachelors_or_higher','category':'education','sql_expr':'s1501.s1501_c02_015e','source_view':SOURCE_VIEWS['s1501'],'source_columns':['s1501_c02_015e'],'definition_policy':'official_field','numerator':'population 25+ bachelor degree or higher','denominator':'population 25 years and over','include_in_cluster_input':True,'notes':'Official S1501 percentage'},

    {'metric_name':'poverty_base_population','category':'poverty','sql_expr':'s1701.s1701_c01_001e','source_view':SOURCE_VIEWS['s1701'],'source_columns':['s1701_c01_001e'],'definition_policy':'official_field','numerator':'population for whom poverty status is determined','denominator':None,'include_in_cluster_input':False,'notes':'Official poverty-status base count from S1701'},
    {'metric_name':'poverty_below_count','category':'poverty','sql_expr':'s1701.s1701_c02_001e','source_view':SOURCE_VIEWS['s1701'],'source_columns':['s1701_c02_001e'],'definition_policy':'official_field','numerator':'population below poverty','denominator':None,'include_in_cluster_input':False,'notes':'Official below-poverty count from S1701'},
    {'metric_name':'poverty_rate','category':'poverty','sql_expr':'s1701.s1701_c03_001e','source_view':SOURCE_VIEWS['s1701'],'source_columns':['s1701_c03_001e'],'definition_policy':'official_field','numerator':'population below poverty','denominator':'population for whom poverty status is determined','include_in_cluster_input':True,'notes':'Official poverty percentage from S1701'},
    {'metric_name':'labor_force_participation_rate','category':'employment','sql_expr':'s2301.s2301_c02_001e','source_view':SOURCE_VIEWS['s2301'],'source_columns':['s2301_c02_001e'],'definition_policy':'official_field','numerator':'civilian labor force participants','denominator':'population 16 years and over','include_in_cluster_input':False,'notes':'Official labor force participation rate from S2301'},
    {'metric_name':'employment_population_ratio','category':'employment','sql_expr':'s2301.s2301_c03_001e','source_view':SOURCE_VIEWS['s2301'],'source_columns':['s2301_c03_001e'],'definition_policy':'official_field','numerator':'employed population','denominator':'population 16 years and over','include_in_cluster_input':False,'notes':'Official employment-population ratio from S2301'},
    {'metric_name':'unemployment_rate','category':'employment','sql_expr':'s2301.s2301_c04_001e','source_view':SOURCE_VIEWS['s2301'],'source_columns':['s2301_c04_001e'],'definition_policy':'official_field','numerator':'unemployed labor force participants','denominator':'civilian labor force','include_in_cluster_input':True,'notes':'Official unemployment rate from S2301'},

    {'metric_name':'total_households_s1901','category':'income_distribution','sql_expr':'s1901.s1901_c01_001e','source_view':SOURCE_VIEWS['s1901'],'source_columns':['s1901_c01_001e'],'definition_policy':'official_field','numerator':'total households','denominator':None,'include_in_cluster_input':False,'notes':'Official total households from S1901'},
    {'metric_name':'pct_hh_income_under_25k','category':'income_distribution','sql_expr':'ROUND(COALESCE(s1901.s1901_c01_002e, 0) + COALESCE(s1901.s1901_c01_003e, 0) + COALESCE(s1901.s1901_c01_004e, 0), 2)','source_view':SOURCE_VIEWS['s1901'],'source_columns':['s1901_c01_002e','s1901_c01_003e','s1901_c01_004e'],'definition_policy':'derived_from_official_fields','numerator':'households with income under 25k','denominator':'total households','include_in_cluster_input':True,'notes':'Bundle of official S1901 household income-share bins'},
    {'metric_name':'pct_hh_income_25k_50k','category':'income_distribution','sql_expr':'ROUND(COALESCE(s1901.s1901_c01_005e, 0) + COALESCE(s1901.s1901_c01_006e, 0), 2)','source_view':SOURCE_VIEWS['s1901'],'source_columns':['s1901_c01_005e','s1901_c01_006e'],'definition_policy':'derived_from_official_fields','numerator':'households with income 25k to 49,999','denominator':'total households','include_in_cluster_input':True,'notes':'Bundle of official S1901 household income-share bins'},
    {'metric_name':'pct_hh_income_50k_100k','category':'income_distribution','sql_expr':'ROUND(COALESCE(s1901.s1901_c01_007e, 0) + COALESCE(s1901.s1901_c01_008e, 0), 2)','source_view':SOURCE_VIEWS['s1901'],'source_columns':['s1901_c01_007e','s1901_c01_008e'],'definition_policy':'derived_from_official_fields','numerator':'households with income 50k to 99,999','denominator':'total households','include_in_cluster_input':True,'notes':'Bundle of official S1901 household income-share bins'},
    {'metric_name':'pct_hh_income_100k_plus','category':'income_distribution','sql_expr':'ROUND(COALESCE(s1901.s1901_c01_009e, 0) + COALESCE(s1901.s1901_c01_010e, 0) + COALESCE(s1901.s1901_c01_011e, 0), 2)','source_view':SOURCE_VIEWS['s1901'],'source_columns':['s1901_c01_009e','s1901_c01_010e','s1901_c01_011e'],'definition_policy':'derived_from_official_fields','numerator':'households with income 100k or more','denominator':'total households','include_in_cluster_input':True,'notes':'Bundle of official S1901 household income-share bins'},
    {'metric_name':'median_household_income_s1901','category':'income_distribution','sql_expr':'s1901.s1901_c01_012e','source_view':SOURCE_VIEWS['s1901'],'source_columns':['s1901_c01_012e'],'definition_policy':'official_field','numerator':'median household income','denominator':None,'include_in_cluster_input':False,'notes':'Official median household income from S1901'},
    {'metric_name':'mean_household_income_s1901','category':'income_distribution','sql_expr':'s1901.s1901_c01_013e','source_view':SOURCE_VIEWS['s1901'],'source_columns':['s1901_c01_013e'],'definition_policy':'official_field','numerator':'mean household income','denominator':None,'include_in_cluster_input':False,'notes':'Official mean household income from S1901'},

    {'metric_name':'language_population_5_plus','category':'language','sql_expr':'s1601.s1601_c01_001e','source_view':SOURCE_VIEWS['s1601'],'source_columns':['s1601_c01_001e'],'definition_policy':'official_field','numerator':'population age 5 and over','denominator':None,'include_in_cluster_input':False,'notes':'Official population 5+ base from S1601'},
    {'metric_name':'speak_language_other_than_english_at_home_count','category':'language','sql_expr':'s1601.s1601_c01_003e','source_view':SOURCE_VIEWS['s1601'],'source_columns':['s1601_c01_003e'],'definition_policy':'official_field','numerator':'population 5+ speaking a language other than English at home','denominator':None,'include_in_cluster_input':False,'notes':'Official count from S1601'},
    {'metric_name':'pct_speak_language_other_than_english_at_home','category':'language','sql_expr':'s1601.s1601_c02_003e','source_view':SOURCE_VIEWS['s1601'],'source_columns':['s1601_c02_003e'],'definition_policy':'official_field','numerator':'population 5+ speaking a language other than English at home','denominator':'population age 5 and over','include_in_cluster_input':True,'notes':'Official percentage from S1601'},
]

lineage_df = pd.DataFrame(metric_specs)
print(f'[OK] Defined {len(lineage_df)} audited metric specs')
lineage_df.head(10)


[OK] Defined 45 audited metric specs


,metric_name,category,sql_expr,source_view,source_columns,definition_policy,numerator,denominator,include_in_cluster_input,notes
0,age_under_18_population,age,s0101.s0101_c01_022e,vw_int_acs_s0101_all_years,[s0101_c01_022e],official_field,population under 18,NaN,False,Official selected age category count from S0101
1,age_18_24_population,age,s0101.s0101_c01_023e,vw_int_acs_s0101_all_years,[s0101_c01_023e],official_field,population age 18 to 24,NaN,False,Official selected age category count from S0101
2,age_65_plus_population,age,s0101.s0101_c01_030e,vw_int_acs_s0101_all_years,[s0101_c01_030e],official_field,population age 65 and over,NaN,False,Official selected age category count from S0101
3,pct_age_under_18,age,s0101.s0101_c02_022e,vw_int_acs_s0101_all_years,[s0101_c02_022e],official_field,population under 18,total population,True,Official selected age category percent from S0101
4,pct_age_18_24,age,s0101.s0101_c02_023e,vw_int_acs_s0101_all_years,[s0101_c02_023e],official_field,population age 18 to 24,total population,True,Official selected age category percent from S0101
5,pct_age_65_plus,age,s0101.s0101_c02_030e,vw_int_acs_s0101_all_years,[s0101_c02_030e],official_field,population age 65 and over,total population,True,Official selected age category percent from S0101
6,median_age_years,age,s0101.s0101_c01_032e,vw_int_acs_s0101_all_years,[s0101_c01_032e],official_field,median age,NaN,False,Official summary indicator from S0101
7,total_households_s1101,household_composition,s1101.s1101_c01_001e,vw_int_acs_s1101_all_years,[s1101_c01_001e],official_field,total households,NaN,False,Official total households from S1101
8,avg_household_size_v2,household_composition,s1101.s1101_c01_002e,vw_int_acs_s1101_all_years,[s1101_c01_002e],official_field,average household size,NaN,True,Official average household size from S1101
9,total_families_s1101,household_composition,s1101.s1101_c01_003e,vw_int_acs_s1101_all_years,[s1101_c01_003e],official_field,total families,NaN,False,Official family count from S1101


In [8]:
required_columns = {}
for spec in metric_specs:
    if spec['source_view'] is None:
        continue
    required_columns.setdefault(spec['source_view'], set()).update(spec['source_columns'])

missing_by_view = {}
for view_name, cols_needed in required_columns.items():
    actual = set(get_columns(view_name, DB_SCHEMA))
    missing = sorted(cols_needed - actual)
    if missing:
        missing_by_view[view_name] = missing

if missing_by_view:
    raise ValueError(json.dumps(missing_by_view, indent=2))

print('[OK] All audited source columns were found in the int views')


[OK] All audited source columns were found in the int views


In [9]:
TARGET_TABLE = 'fact_acs_tract_profile_v2'
TARGET_VIEWS = {
    'all_years': 'vw_acs_tract_profile_v2_all_years',
    'year_2023': 'vw_acs_tract_profile_v2_2023',
    'stable_4yr': 'vw_acs_tract_profile_v2_stable_4yr',
    'cluster_input': 'vw_acs_tract_profile_v2_cluster_input',
}
SQL_FILE = SQL_DIR / '07c_rebuild_fact_acs_tract_profile_v2_audited.sql'


In [10]:
def build_metric_select_sql(specs: list[dict]) -> str:
    lines = []
    for spec in specs:
        lines.append(f"    {spec['sql_expr']} AS {spec['metric_name']}")
    return ',\n'.join(lines)

metric_select_sql = build_metric_select_sql(metric_specs)


In [11]:
sql_text = f'''
DROP VIEW IF EXISTS {DB_SCHEMA}.{TARGET_VIEWS['cluster_input']};
DROP VIEW IF EXISTS {DB_SCHEMA}.{TARGET_VIEWS['stable_4yr']};
DROP VIEW IF EXISTS {DB_SCHEMA}.{TARGET_VIEWS['year_2023']};
DROP VIEW IF EXISTS {DB_SCHEMA}.{TARGET_VIEWS['all_years']};
DROP TABLE IF EXISTS {DB_SCHEMA}.{TARGET_TABLE};

CREATE TABLE {DB_SCHEMA}.{TARGET_TABLE} AS
SELECT
    p.*,
{metric_select_sql}
FROM {DB_SCHEMA}.{BASE_RELATION} p
LEFT JOIN {DB_SCHEMA}.{SOURCE_VIEWS['s0101']} s0101
  ON p.year = s0101.year AND p.tract_geoid = s0101.tract_geoid
LEFT JOIN {DB_SCHEMA}.{SOURCE_VIEWS['s1101']} s1101
  ON p.year = s1101.year AND p.tract_geoid = s1101.tract_geoid
LEFT JOIN {DB_SCHEMA}.{SOURCE_VIEWS['s1501']} s1501
  ON p.year = s1501.year AND p.tract_geoid = s1501.tract_geoid
LEFT JOIN {DB_SCHEMA}.{SOURCE_VIEWS['s1701']} s1701
  ON p.year = s1701.year AND p.tract_geoid = s1701.tract_geoid
LEFT JOIN {DB_SCHEMA}.{SOURCE_VIEWS['s1901']} s1901
  ON p.year = s1901.year AND p.tract_geoid = s1901.tract_geoid
LEFT JOIN {DB_SCHEMA}.{SOURCE_VIEWS['s2301']} s2301
  ON p.year = s2301.year AND p.tract_geoid = s2301.tract_geoid
LEFT JOIN {DB_SCHEMA}.{SOURCE_VIEWS['s1601']} s1601
  ON p.year = s1601.year AND p.tract_geoid = s1601.tract_geoid
;

CREATE VIEW {DB_SCHEMA}.{TARGET_VIEWS['all_years']} AS
SELECT *
FROM {DB_SCHEMA}.{TARGET_TABLE}
ORDER BY year, tract_geoid;

CREATE VIEW {DB_SCHEMA}.{TARGET_VIEWS['year_2023']} AS
SELECT *
FROM {DB_SCHEMA}.{TARGET_TABLE}
WHERE year = 2023
ORDER BY tract_geoid;

CREATE VIEW {DB_SCHEMA}.{TARGET_VIEWS['stable_4yr']} AS
SELECT *
FROM {DB_SCHEMA}.{TARGET_TABLE}
WHERE is_stable_all_4_years = 1
ORDER BY year, tract_geoid;

CREATE VIEW {DB_SCHEMA}.{TARGET_VIEWS['cluster_input']} AS
SELECT
    year,
    tract_geoid,
    tract_name_latest,
    is_stable_all_4_years,
    median_household_income,
    pct_rent_burden_30_plus,
    pct_rent_burden_50_plus,
    pct_renter_occupied,
    pct_vacant_housing_units,
    total_population,
    pct_white_non_hispanic,
    pct_black_non_hispanic,
    pct_hispanic,
    poverty_rate,
    unemployment_rate,
    pct_hh_income_under_25k,
    pct_hh_income_25k_50k,
    pct_hh_income_50k_100k,
    pct_hh_income_100k_plus,
    pct_less_than_high_school,
    pct_some_college_or_associate,
    pct_bachelors_or_higher,
    pct_age_under_18,
    pct_age_18_24,
    pct_age_65_plus,
    avg_household_size_v2,
    pct_family_households,
    pct_households_with_own_children_under_18,
    pct_households_with_under_18,
    pct_households_with_65_plus,
    pct_one_person_households,
    pct_speak_language_other_than_english_at_home
FROM {DB_SCHEMA}.{TARGET_TABLE}
ORDER BY year, tract_geoid;
'''

SQL_FILE.write_text(sql_text, encoding='utf-8')
print(f'[OK] SQL written to: {SQL_FILE}')
print(sql_text[:2500])


[OK] SQL written to: D:\Projects\Community-Pulse\sql\acs\fact\07c_rebuild_fact_acs_tract_profile_v2_audited.sql

DROP VIEW IF EXISTS public.vw_acs_tract_profile_v2_cluster_input;
DROP VIEW IF EXISTS public.vw_acs_tract_profile_v2_stable_4yr;
DROP VIEW IF EXISTS public.vw_acs_tract_profile_v2_2023;
DROP VIEW IF EXISTS public.vw_acs_tract_profile_v2_all_years;
DROP TABLE IF EXISTS public.fact_acs_tract_profile_v2;

CREATE TABLE public.fact_acs_tract_profile_v2 AS
SELECT
    p.*,
    s0101.s0101_c01_022e AS age_under_18_population,
    s0101.s0101_c01_023e AS age_18_24_population,
    s0101.s0101_c01_030e AS age_65_plus_population,
    s0101.s0101_c02_022e AS pct_age_under_18,
    s0101.s0101_c02_023e AS pct_age_18_24,
    s0101.s0101_c02_030e AS pct_age_65_plus,
    s0101.s0101_c01_032e AS median_age_years,
    s1101.s1101_c01_001e AS total_households_s1101,
    s1101.s1101_c01_002e AS avg_household_size_v2,
    s1101.s1101_c01_003e AS total_families_s1101,
    CASE WHEN s1101.s1101_c01_

In [12]:
RUN_SQL = True
if RUN_SQL:
    execute_sql(sql_text)
    print(f'[OK] Rebuilt {DB_SCHEMA}.{TARGET_TABLE}')
else:
    print('[INFO] SQL generated only, not executed')


[OK] Rebuilt public.fact_acs_tract_profile_v2


In [13]:
lineage_out = lineage_df.copy()
lineage_out['source_columns'] = lineage_out['source_columns'].apply(lambda x: ', '.join(x))
lineage_out.to_csv(OUTPUT_DIR / 'v2_audited_metric_lineage.csv', index=False)

pd.DataFrame({'cluster_metric_name': [
    'median_household_income',
    'pct_rent_burden_30_plus',
    'pct_rent_burden_50_plus',
    'pct_renter_occupied',
    'pct_vacant_housing_units',
    'poverty_rate',
    'unemployment_rate',
    'pct_hh_income_under_25k',
    'pct_hh_income_25k_50k',
    'pct_hh_income_50k_100k',
    'pct_hh_income_100k_plus',
    'pct_less_than_high_school',
    'pct_some_college_or_associate',
    'pct_bachelors_or_higher',
    'pct_age_under_18',
    'pct_age_18_24',
    'pct_age_65_plus',
    'avg_household_size_v2',
    'pct_family_households',
    'pct_households_with_own_children_under_18',
    'pct_households_with_under_18',
    'pct_households_with_65_plus',
    'pct_one_person_households',
    'pct_speak_language_other_than_english_at_home',
]}).to_csv(OUTPUT_DIR / 'v2_cluster_metric_list.csv', index=False)

print('[OK] Wrote metric lineage and cluster metric list')


[OK] Wrote metric lineage and cluster metric list


In [14]:
inventory_query = f'''
SELECT column_name
FROM information_schema.columns
WHERE table_schema = '{DB_SCHEMA}'
  AND table_name = '{TARGET_TABLE}'
ORDER BY ordinal_position
'''
inventory_df = read_sql_df(inventory_query)
inventory_df.to_csv(OUTPUT_DIR / 'v2_metric_inventory.csv', index=False)

sample_df = read_sql_df(f'''
SELECT *
FROM {DB_SCHEMA}.{TARGET_TABLE}
ORDER BY year, tract_geoid
LIMIT 10
''')
sample_df.to_csv(OUTPUT_DIR / 'fact_acs_tract_profile_v2_sample.csv', index=False)

core_metrics = [
    'poverty_rate',
    'unemployment_rate',
    'pct_hh_income_under_25k',
    'pct_bachelors_or_higher',
    'pct_age_18_24',
    'pct_family_households',
    'pct_households_with_under_18',
    'pct_households_with_65_plus',
]

null_rows = []
for metric in core_metrics:
    q = f'''
    SELECT
        year,
        COUNT(*) AS row_count,
        SUM(CASE WHEN {metric} IS NULL THEN 1 ELSE 0 END) AS null_count,
        ROUND(100.0 * SUM(CASE WHEN {metric} IS NULL THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS null_pct,
        ROUND(AVG({metric})::numeric, 2) AS avg_value
    FROM {DB_SCHEMA}.{TARGET_TABLE}
    GROUP BY year
    ORDER BY year
    '''
    tmp = read_sql_df(q)
    tmp.insert(1, 'metric_name', metric)
    null_rows.append(tmp)

null_profile_df = pd.concat(null_rows, ignore_index=True)
null_profile_df.to_csv(OUTPUT_DIR / 'fact_acs_tract_profile_v2_null_profile.csv', index=False)

summary_df = read_sql_df(f'''
WITH yearly AS (
    SELECT
        year,
        COUNT(*) AS row_count,
        SUM(CASE WHEN poverty_rate IS NOT NULL
                  AND unemployment_rate IS NOT NULL
                  AND pct_hh_income_under_25k IS NOT NULL
                  AND pct_bachelors_or_higher IS NOT NULL
                  AND pct_age_18_24 IS NOT NULL
                  AND pct_family_households IS NOT NULL
             THEN 1 ELSE 0 END) AS rows_with_v2_core_metrics,
        ROUND(AVG(poverty_rate)::numeric, 2) AS avg_poverty_rate,
        ROUND(AVG(unemployment_rate)::numeric, 2) AS avg_unemployment_rate,
        ROUND(AVG(pct_hh_income_under_25k)::numeric, 2) AS avg_pct_hh_income_under_25k,
        ROUND(AVG(pct_bachelors_or_higher)::numeric, 2) AS avg_pct_bachelors_or_higher,
        ROUND(AVG(pct_age_18_24)::numeric, 2) AS avg_pct_age_18_24,
        ROUND(AVG(pct_family_households)::numeric, 2) AS avg_pct_family_households
    FROM {DB_SCHEMA}.{TARGET_TABLE}
    GROUP BY year
)
SELECT *
FROM yearly
ORDER BY year
''')
summary_df.to_csv(OUTPUT_DIR / 'v2_run_summary.csv', index=False)

print('[OK] Exported inventory, sample, null profile, and run summary')
inventory_df.head(), summary_df


[OK] Exported inventory, sample, null profile, and run summary


(   column_name
 0         year
 1  tract_geoid
 2       geo_id
 3      statefp
 4     countyfp,
    year  row_count  rows_with_v2_core_metrics  avg_poverty_rate  \
 0  2019         43                         43             24.38   
 1  2021         48                         48             22.75   
 2  2022         48                         48             22.90   
 3  2023         48                         48             22.27   
 
    avg_unemployment_rate  avg_pct_hh_income_under_25k  \
 0                   5.07                        30.09   
 1                   5.01                        27.49   
 2                   4.71                        25.40   
 3                   4.40                        24.74   
 
    avg_pct_bachelors_or_higher  avg_pct_age_18_24  avg_pct_family_households  
 0                        47.59              24.04                      48.96  
 1                        49.38              23.95                      49.05  
 2                        49.

In [15]:
audit_query = f'''
SELECT
    year,
    MAX(ABS(pct_family_households - CASE WHEN total_households_s1101 > 0 THEN ROUND(100.0 * total_families_s1101 / total_households_s1101, 2) ELSE NULL END)) AS max_diff_pct_family_households,
    MAX(ABS(pct_households_with_own_children_under_18 - CASE WHEN total_households_s1101 > 0 THEN ROUND(100.0 * households_with_own_children_under_18 / total_households_s1101, 2) ELSE NULL END)) AS max_diff_pct_households_with_own_children_under_18,
    MAX(ABS(pct_less_than_high_school - CASE WHEN pct_high_school_or_higher IS NOT NULL THEN ROUND(100.0 - pct_high_school_or_higher, 2) ELSE NULL END)) AS max_diff_pct_less_than_high_school,
    MAX(ABS(pct_some_college_or_associate - ROUND(COALESCE(pct_some_college_no_degree, 0) + COALESCE(pct_associates_degree, 0), 2))) AS max_diff_pct_some_college_or_associate,
    MAX(ABS(pct_hh_income_under_25k + pct_hh_income_25k_50k + pct_hh_income_50k_100k + pct_hh_income_100k_plus - 100.0)) AS max_diff_income_groups_sum_to_100
FROM {DB_SCHEMA}.{TARGET_TABLE}
GROUP BY year
ORDER BY year
'''

audit_df = read_sql_df(audit_query)
audit_df.to_csv(OUTPUT_DIR / 'v2_formula_audit_summary.csv', index=False)
audit_df


,year,max_diff_pct_family_households,max_diff_pct_households_with_own_children_under_18,max_diff_pct_less_than_high_school,max_diff_pct_some_college_or_associate,max_diff_income_groups_sum_to_100
0,2019,0.0,0.0,0.0,0.0,0.2
1,2021,0.0,0.0,0.0,0.0,0.2
2,2022,0.0,0.0,0.0,0.0,0.2
3,2023,0.0,0.0,0.0,0.0,0.3


In [16]:
print('[DONE] Audited V2 rebuild complete')
print(f'  - Table: {DB_SCHEMA}.{TARGET_TABLE}')
print(f'  - SQL file: {SQL_FILE}')
print(f'  - Lineage file: {OUTPUT_DIR / "v2_audited_metric_lineage.csv"}')
print(f'  - Cluster metric list: {OUTPUT_DIR / "v2_cluster_metric_list.csv"}')
print(f'  - Summary file: {OUTPUT_DIR / "v2_run_summary.csv"}')
print(f'  - Formula audit file: {OUTPUT_DIR / "v2_formula_audit_summary.csv"}')


[DONE] Audited V2 rebuild complete
  - Table: public.fact_acs_tract_profile_v2
  - SQL file: D:\Projects\Community-Pulse\sql\acs\fact\07c_rebuild_fact_acs_tract_profile_v2_audited.sql
  - Lineage file: D:\Projects\Community-Pulse\outputs\acs\analysis\base\v2_audited_metric_lineage.csv
  - Cluster metric list: D:\Projects\Community-Pulse\outputs\acs\analysis\base\v2_cluster_metric_list.csv
  - Summary file: D:\Projects\Community-Pulse\outputs\acs\analysis\base\v2_run_summary.csv
  - Formula audit file: D:\Projects\Community-Pulse\outputs\acs\analysis\base\v2_formula_audit_summary.csv
